In [ ]:
import pandas as pd, glob, os

dome = pd.read_csv("../data/AMI/dome_annotations_M1.csv", sep=";", skipinitialspace=True)
dome["name"] = dome["name"].str.strip()
print(dome.shape)
print("Meetings in DOME:", dome["name"].nunique())
print(dome.head(3).to_string())

word_files = glob.glob("../data/AMI/manual_1.6.2/words/*.words.xml")
meetings_with_words = {os.path.basename(f).split(".")[0] for f in word_files}
print("Meetings with word files:", len(meetings_with_words))
print("DOME meetings that have word files:",
      len(set(dome["name"]) & meetings_with_words), "of", dome["name"].nunique())

In [ ]:
import re, glob
from collections import Counter

base = "../data/AMI/manual_1.6.2"

def show(path, n=25):
    print("=" * 80)
    print(path)
    print("=" * 80)
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line.rstrip()[:220])

seg = sorted(glob.glob(f"{base}/segments/ES2002a.*.segments.xml"))[0]
wrd = sorted(glob.glob(f"{base}/words/ES2002a.*.words.xml"))[0]

show(f"{base}/corpusResources/meetings.xml", 30)
show(f"{base}/corpusResources/participants.xml", 20)
show(seg, 20)
show(wrd, 25)

txt = open(f"{base}/corpusResources/participants.xml", encoding="utf-8").read()
print("gender values:", Counter(re.findall(r'gender="([^"]*)"', txt)))

In [ ]:
import re, os
import xml.etree.ElementTree as ET
import pandas as pd

BASE = "../data/AMI/manual_1.6.2"
NITE = "{http://nite.sourceforge.net/}"

# 1) meetings.xml: (meeting, seat letter) -> person id and role
rows = []
root = ET.parse(f"{BASE}/corpusResources/meetings.xml").getroot()
for m in root.iter("meeting"):
    for s in m.iter("speaker"):
        rows.append({"meeting": m.get("observation"),
                     "agent": s.get("nxt_agent"),
                     "person": s.get("global_name"),
                     "role": s.get("role")})
speakers = pd.DataFrame(rows)

# 2) participants.xml: person id -> sex
rows = []
root = ET.parse(f"{BASE}/corpusResources/participants.xml").getroot()
for p in root.iter("participant"):
    rows.append({"person": p.get(NITE + "id"), "sex": p.get("sex")})
people = pd.DataFrame(rows).drop_duplicates("person")

speakers = speakers.merge(people, on="person", how="left")
speakers["gender"] = speakers["sex"].map({"F": "F", "M": "M"})
print("Speaker-seat rows:", len(speakers), "| meetings:", speakers["meeting"].nunique())
print("Seats with no sex found:", speakers["sex"].isna().sum())

# 3) words file -> flag whether each item is a real word (not punctuation)
HREF = re.compile(r"#id\(([^)]+)\)(?:\.\.id\(([^)]+)\))?")

def read_words(path):
    items, index = [], {}
    for el in ET.parse(path).getroot():
        index[el.get(NITE + "id")] = len(items)
        items.append(el.tag == "w" and el.get("punc") != "true")
    return items, index

# 4) segments file -> one row per utterance (segment) with word count
def read_segments(meeting, agent):
    seg_path = f"{BASE}/segments/{meeting}.{agent}.segments.xml"
    wrd_path = f"{BASE}/words/{meeting}.{agent}.words.xml"
    if not (os.path.exists(seg_path) and os.path.exists(wrd_path)):
        return []
    items, index = read_words(wrd_path)
    out = []
    for seg in ET.parse(seg_path).getroot().iter("segment"):
        n_words = 0
        for ch in seg.iter(NITE + "child"):
            mt = HREF.search(ch.get("href", ""))
            if not mt:
                continue
            a = index.get(mt.group(1))
            b = index.get(mt.group(2)) if mt.group(2) else a
            if a is None or b is None:
                continue
            n_words += sum(items[a:b + 1])
        out.append({"start": float(seg.get("transcriber_start")),
                    "end": float(seg.get("transcriber_end")),
                    "n_words": n_words})
    return out

recs = []
for r in speakers.itertuples():
    for k, s in enumerate(read_segments(r.meeting, r.agent)):
        recs.append({"corpus": "AMI", "meeting": r.meeting, "series": r.meeting[:-1],
                     "agent": r.agent, "speaker": r.person, "role": r.role,
                     "gender": r.gender, "utterance_id": k, **s})
ami = pd.DataFrame(recs)
ami["dur"] = ami["end"] - ami["start"]

os.makedirs("../outputs", exist_ok=True)
ami.to_csv("../outputs/ami_segments_raw.csv", index=False)

# ---- checks ----
print("\nRows:", ami.shape)
print("Meetings:", ami["meeting"].nunique(), "| series:", ami["series"].nunique(),
      "| persons:", ami["speaker"].nunique())
print("\nUtterance rows by gender:")
print(ami["gender"].value_counts(dropna=False))
print("Unmapped gender share:", round(ami["gender"].isna().mean(), 3))
print("\nPersons by gender:",
      ami.drop_duplicates("speaker")["gender"].value_counts(dropna=False).to_dict())
print("\nTotal speaking time (s) and utterances by gender:")
print(ami.groupby("gender")["dur"].agg(["sum", "size"]).round(1))
print("\nDuration summary:")
print(ami["dur"].describe())
print("dur <= 0:", (ami["dur"] <= 0).sum(), "| dur > 60:", (ami["dur"] > 60).sum())
print("Share of 1-2 word utterances:", round((ami["n_words"] <= 2).mean(), 3))

ms = ami.drop_duplicates(["meeting", "speaker"])
print("\nRole x gender (scenario meetings only have roles):")
print(pd.crosstab(ms["role"], ms["gender"]))

In [ ]:
import numpy as np

# --- 1. Can the person ID prefix (F.../M...) be trusted as gender? ---
known = people.dropna(subset=["sex"]).copy()
known["prefix_gender"] = known["person"].str[0]
agree = (known["prefix_gender"] == known["sex"]).mean()
print("Persons with sex in participants.xml:", len(known),
      "| ID prefix agrees:", round(agree, 4))
print(known[known["prefix_gender"] != known["sex"]].head().to_string())

unk = ami[ami["gender"].isna()].drop_duplicates("speaker")["speaker"]
print("\nUnmapped persons:", len(unk), "| first letters:", unk.str[0].value_counts().to_dict())
print(unk.head(10).tolist())

# --- 2. Fill gender from the ID prefix ONLY if the check above is (almost) perfect ---
ami["gender_source"] = np.where(ami["gender"].notna(), "participants.xml", "unknown")
if agree >= 0.99:
    fill = ami["speaker"].str[0].map({"F": "F", "M": "M"})
    use = ami["gender"].isna() & fill.notna()
    ami.loc[use, "gender"] = fill[use]
    ami.loc[use, "gender_source"] = "id_prefix"
print("\nGender after fill:")
print(ami["gender"].value_counts(dropna=False))
print("Unmapped share now:", round(ami["gender"].isna().mean(), 3))

# --- 3. Odd durations ---
print("\nRows with dur <= 0:")
print(ami[ami["dur"] <= 0][["meeting", "speaker", "start", "end", "dur", "n_words"]].to_string())
long_ = ami[ami["dur"] > 60][["meeting", "speaker", "start", "end", "dur", "n_words"]].copy()
long_["sec_per_word"] = (long_["dur"] / long_["n_words"].replace(0, np.nan)).round(2)
print("\nLongest rows:")
print(long_.sort_values("dur", ascending=False).head(12).to_string())

# --- 4. Are people shared across series? Series composition ---
print("\nPersons appearing in more than one series:",
      int((ami.groupby("speaker")["series"].nunique() > 1).sum()))
pers = ami.dropna(subset=["gender"]).drop_duplicates(["series", "speaker"])
comp = (pers.groupby(["series", "gender"]).size().unstack(fill_value=0)
        .reindex(columns=["F", "M"], fill_value=0))
print("Series:", len(comp), "| with both genders:", int(((comp["F"] > 0) & (comp["M"] > 0)).sum()))
print(comp.value_counts().head(8))

# --- 5. Backchannels and parity ---
ami["short"] = ami["n_words"] <= 2
print("\nShare of 1-2 word utterances by gender:")
print(ami.groupby("gender")["short"].mean().round(3))
t = ami.groupby("gender")["dur"].sum()
pp = ami.drop_duplicates("speaker")["gender"].value_counts()
print("\nF time share:", round(t["F"] / t.sum(), 3), "| F share of persons:", round(pp["F"] / pp.sum(), 3))

In [ ]:
import numpy as np
import pandas as pd

# ---------- 0. Is the ID-prefix rule safe for the 40 unmapped persons? ----------
print("2nd character among persons WITH sex:", known["person"].str[1].value_counts().to_dict())
print("2nd character among the 40 unmapped :", unk.str[1].value_counts().to_dict())

# ---------- 1. Clean ----------
bad = (ami["dur"] <= 0) | ((ami["n_words"] == 0) & (ami["dur"] > 10))
print("\nRemoved rows:", int(bad.sum()))
print(ami[bad][["meeting", "speaker", "start", "end", "dur", "n_words"]].to_string())
a = ami[~bad].copy()
a["backchannel"] = a["n_words"] <= 2
print("Backchannel share of segments:", round(a["backchannel"].mean(), 3),
      "| of speaking time:", round(a.loc[a["backchannel"], "dur"].sum() / a["dur"].sum(), 3))

# ---------- 2. Build turns (>= 3 words, merge same-speaker segments with gap <= 1 s) ----------
GAP = 1.0
KEEP = ["corpus", "meeting", "series", "speaker", "role", "gender", "gender_source"]

def build_turns(df, gap=GAP):
    out = []
    for meeting, d in df[~df["backchannel"]].groupby("meeting"):
        for spk, s in d.groupby("speaker"):
            others = np.sort(d.loc[d["speaker"] != spk, "start"].to_numpy())
            cur = None
            for r in s.sort_values("start").itertuples():
                if cur is not None:
                    i0 = np.searchsorted(others, cur["end"], side="right")
                    i1 = np.searchsorted(others, r.start, side="left")
                    if (r.start - cur["end"]) <= gap and i1 <= i0:
                        cur["end"] = max(cur["end"], r.end)
                        cur["n_words"] += r.n_words
                        continue
                    out.append(cur)
                cur = {**{k: getattr(r, k) for k in KEEP},
                       "start": r.start, "end": r.end, "n_words": r.n_words}
            if cur is not None:
                out.append(cur)
    t = pd.DataFrame(out)
    t["dur"] = t["end"] - t["start"]
    return t

turns = build_turns(a)
print("\nTurns:", len(turns), "from", int((~a["backchannel"]).sum()), "segments")
print(turns.groupby("gender")["dur"].agg(time="sum", turns="size", mean_len="mean").round(2))
print(turns["dur"].describe().round(2))

# ---------- 3. Person-connected components (identity firewall) ----------
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

for m, d in speakers.groupby("meeting"):
    ps = d["person"].dropna().tolist()
    for p in ps[1:]:
        ra, rb = find(ps[0]), find(p)
        if ra != rb:
            parent[rb] = ra

roots = sorted({find(p) for p in speakers["person"].dropna()})
comp_id = {r: f"AMI_{i:03d}" for i, r in enumerate(roots, 1)}
turns["component"] = turns["speaker"].map(lambda p: comp_id[find(p)])

# ---------- 4. Meeting-level table ----------
g = (turns.groupby(["meeting", "component", "gender"])
          .agg(T=("dur", "sum"), N=("dur", "size"), n=("speaker", "nunique"))
          .reset_index())
w = g.pivot(index=["meeting", "component"], columns="gender", values=["T", "N", "n"]).fillna(0)
mi = pd.DataFrame({"T_F": w["T"]["F"], "T_M": w["T"]["M"],
                   "N_F": w["N"]["F"], "N_M": w["N"]["M"],
                   "n_F": w["n"]["F"], "n_M": w["n"]["M"]}).reset_index()
mi["mixed"] = (mi["n_F"] > 0) & (mi["n_M"] > 0)
pref = turns[turns["gender_source"] == "id_prefix"].groupby("meeting").size()
mi["has_prefix_gender"] = mi["meeting"].isin(pref.index)

print("\nMeetings:", len(mi), "| mixed-gender:", int(mi["mixed"].sum()))
print("Mixed meetings if id-prefix persons are excluded:",
      int((mi["mixed"] & ~mi["has_prefix_gender"]).sum()))
print("Components:", mi["component"].nunique(),
      "| with a mixed meeting:", mi.loc[mi["mixed"], "component"].nunique())
print("Largest components (meetings):", mi["component"].value_counts().head(5).to_dict())

mm = mi[mi["mixed"]].copy()
mm["log_parity_time"] = np.log((mm["T_F"] / mm["n_F"]) / (mm["T_M"] / mm["n_M"]))
mm["log_parity_turns"] = np.log((mm["N_F"] / mm["n_F"]) / (mm["N_M"] / mm["n_M"]))
print("\nPer-person parity in mixed meetings (log scale, 0 = equal):")
print(mm[["log_parity_time", "log_parity_turns"]].describe().round(3))
print("Median per-person time parity (F/M):", round(float(np.exp(mm["log_parity_time"].median())), 3))

turns.to_csv("../outputs/ami_turns.csv", index=False)
mi.to_csv("../outputs/ami_meeting_info.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd

# ---------- 0. Is the ID-prefix rule safe for the 40 unmapped persons? ----------
print("2nd character among persons WITH sex:", known["person"].str[1].value_counts().to_dict())
print("2nd character among the 40 unmapped :", unk.str[1].value_counts().to_dict())

# ---------- 1. Clean ----------
bad = (ami["dur"] <= 0) | ((ami["n_words"] == 0) & (ami["dur"] > 10))
print("\nRemoved rows:", int(bad.sum()))
print(ami[bad][["meeting", "speaker", "start", "end", "dur", "n_words"]].to_string())
a = ami[~bad].copy()
a["backchannel"] = a["n_words"] <= 2
print("Backchannel share of segments:", round(a["backchannel"].mean(), 3),
      "| of speaking time:", round(a.loc[a["backchannel"], "dur"].sum() / a["dur"].sum(), 3))

# ---------- 2. Build turns (>= 3 words, merge same-speaker segments with gap <= 1 s) ----------
GAP = 1.0
KEEP = ["corpus", "meeting", "series", "speaker", "role", "gender", "gender_source"]

def build_turns(df, gap=GAP):
    out = []
    for meeting, d in df[~df["backchannel"]].groupby("meeting"):
        for spk, s in d.groupby("speaker"):
            others = np.sort(d.loc[d["speaker"] != spk, "start"].to_numpy())
            cur = None
            for r in s.sort_values("start").itertuples():
                if cur is not None:
                    i0 = np.searchsorted(others, cur["end"], side="right")
                    i1 = np.searchsorted(others, r.start, side="left")
                    if (r.start - cur["end"]) <= gap and i1 <= i0:
                        cur["end"] = max(cur["end"], r.end)
                        cur["n_words"] += r.n_words
                        continue
                    out.append(cur)
                cur = {**{k: getattr(r, k) for k in KEEP},
                       "start": r.start, "end": r.end, "n_words": r.n_words}
            if cur is not None:
                out.append(cur)
    t = pd.DataFrame(out)
    t["dur"] = t["end"] - t["start"]
    return t

turns = build_turns(a)
print("\nTurns:", len(turns), "from", int((~a["backchannel"]).sum()), "segments")
print(turns.groupby("gender")["dur"].agg(time="sum", turns="size", mean_len="mean").round(2))
print(turns["dur"].describe().round(2))

# ---------- 3. Person-connected components (identity firewall) ----------
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

for m, d in speakers.groupby("meeting"):
    ps = d["person"].dropna().tolist()
    for p in ps[1:]:
        ra, rb = find(ps[0]), find(p)
        if ra != rb:
            parent[rb] = ra

roots = sorted({find(p) for p in speakers["person"].dropna()})
comp_id = {r: f"AMI_{i:03d}" for i, r in enumerate(roots, 1)}
turns["component"] = turns["speaker"].map(lambda p: comp_id[find(p)])

# ---------- 4. Meeting-level table ----------
g = (turns.groupby(["meeting", "component", "gender"])
          .agg(T=("dur", "sum"), N=("dur", "size"), n=("speaker", "nunique"))
          .reset_index())
w = g.pivot(index=["meeting", "component"], columns="gender", values=["T", "N", "n"]).fillna(0)
mi = pd.DataFrame({"T_F": w["T"]["F"], "T_M": w["T"]["M"],
                   "N_F": w["N"]["F"], "N_M": w["N"]["M"],
                   "n_F": w["n"]["F"], "n_M": w["n"]["M"]}).reset_index()
mi["mixed"] = (mi["n_F"] > 0) & (mi["n_M"] > 0)
pref = turns[turns["gender_source"] == "id_prefix"].groupby("meeting").size()
mi["has_prefix_gender"] = mi["meeting"].isin(pref.index)

print("\nMeetings:", len(mi), "| mixed-gender:", int(mi["mixed"].sum()))
print("Mixed meetings if id-prefix persons are excluded:",
      int((mi["mixed"] & ~mi["has_prefix_gender"]).sum()))
print("Components:", mi["component"].nunique(),
      "| with a mixed meeting:", mi.loc[mi["mixed"], "component"].nunique())
print("Largest components (meetings):", mi["component"].value_counts().head(5).to_dict())

mm = mi[mi["mixed"]].copy()
mm["log_parity_time"] = np.log((mm["T_F"] / mm["n_F"]) / (mm["T_M"] / mm["n_M"]))
mm["log_parity_turns"] = np.log((mm["N_F"] / mm["n_F"]) / (mm["N_M"] / mm["n_M"]))
print("\nPer-person parity in mixed meetings (log scale, 0 = equal):")
print(mm[["log_parity_time", "log_parity_turns"]].describe().round(3))
print("Median per-person time parity (F/M):", round(float(np.exp(mm["log_parity_time"].median())), 3))

turns.to_csv("../outputs/ami_turns.csv", index=False)
mi.to_csv("../outputs/ami_meeting_info.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd

EPS = 0.2   # seconds; a start must be this far inside another turn to count as a cut-in

def meeting_features(t):
    t = t.sort_values("start")
    st, en = t["start"].to_numpy(), t["end"].to_numpy()
    spk, gen = t["speaker"].to_numpy(), t["gender"].to_numpy()
    dur = t["dur"].to_numpy()
    isF, isM = gen == "F", gen == "M"
    nF, nM = len(set(spk[isF])), len(set(spk[isM]))
    if nF == 0 or nM == 0:
        return None
    TF, TM = dur[isF].sum(), dur[isM].sum()
    NF, NM = int(isF.sum()), int(isM.sum())

    # turn i is cut into when a turn by ANOTHER speaker starts inside it
    inside = ((st[None, :] > st[:, None] + EPS) & (st[None, :] < en[:, None] - EPS)
              & (spk[None, :] != spk[:, None]))
    cutF = int((inside & isF[:, None] & isM[None, :]).sum())   # F turn cut into by an M turn
    cutM = int((inside & isM[:, None] & isF[None, :]).sum())   # M turn cut into by an F turn

    t0, t1 = st.min(), en.max()
    mid = ((st + en) / 2 - t0) / max(t1 - t0, 1e-9)
    cen_F = (mid[isF] * dur[isF]).sum() / TF
    cen_M = (mid[isM] * dur[isM]).sum() / TM

    return {"n_F": nF, "n_M": nM, "N_F": NF, "N_M": NM, "T_F": TF, "T_M": TM,
            "log_parity_time": np.log((TF / nF) / (TM / nM)),
            "log_parity_turns": np.log((NF / nF) / (NM / nM)),
            "log_len_ratio": np.log((TF / NF) / (TM / NM)),
            "interruption_asym": (cutF - cutM) / (NF + NM),
            "position_gap": cen_F - cen_M,
            "cut_F": cutF, "cut_M": cutM}

rows = []
for m, t in turns.groupby("meeting"):
    f = meeting_features(t)
    if f:
        rows.append({"meeting": m, **f})

feat = pd.DataFrame(rows).merge(
    mi[["meeting", "component", "has_prefix_gender"]], on="meeting")
feat["series"] = feat["meeting"].str[:-1]
main = feat[~feat["has_prefix_gender"]].copy()

cols = ["log_parity_time", "log_parity_turns", "log_len_ratio",
        "interruption_asym", "position_gap"]
print("Mixed meetings: all =", len(feat), "| main (no unverified gender) =", len(main))
print("Series in main:", main["series"].nunique(), "| components in main:", main["component"].nunique())
print("Min turns per gender in main:", int(main[["N_F", "N_M"]].min().min()),
      "| meetings with < 10 turns for a gender:", int(((main["N_F"] < 10) | (main["N_M"] < 10)).sum()))

print("\nBaseline (unmodified) feature distribution, main set:")
print(main[cols].describe().round(3).to_string())
print("\nSensitivity: medians, main vs all mixed meetings")
print(pd.DataFrame({"main": main[cols].median(), "all": feat[cols].median()}).round(3))

print("\nCut-in events in main: F turns cut by M =", int(main["cut_F"].sum()),
      "| M turns cut by F =", int(main["cut_M"].sum()),
      "| meetings with zero events:", int(((main["cut_F"] + main["cut_M"]) == 0).sum()))

print("\nSpearman correlations between features (main set):")
print(main[cols].corr(method="spearman").round(2).to_string())

big = turns[turns["dur"] > 120]
print("\nTurns longer than 120 s:", len(big),
      "| share of speaking time:", round(big["dur"].sum() / turns["dur"].sum(), 3))

# save the AMI turns in a MELD-compatible intervals shape, plus the meeting-level features
ami_int = turns.rename(columns={"meeting": "unit_id"}).copy()
ami_int["utterance_id"] = ami_int.groupby("unit_id").cumcount()
ami_int.to_csv("../outputs/ami_intervals.csv", index=False)
feat.to_csv("../outputs/ami_meeting_features.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

def fragment_ami(t, target, level, k=3, gap=1.2, min_piece=0.5, seed=0):
    """Split a fraction `level` of the ELIGIBLE target turns into k pieces (silence `gap` between)."""
    rng = np.random.default_rng(seed)
    t = t.sort_values(["start", "end"]).reset_index(drop=True)
    nxt_own = t.groupby("speaker")["start"].shift(-1)
    room = (nxt_own - t["end"]).fillna(np.inf)
    elig = ((t["gender"] == target) & (t["dur"] >= k * min_piece) & (room >= (k - 1) * gap)).to_numpy()
    chosen = np.zeros(len(t), bool)
    n_pick = int(round(level * elig.sum()))
    if n_pick > 0:
        chosen[rng.choice(np.flatnonzero(elig), size=n_pick, replace=False)] = True
    rows = []
    for i, r in enumerate(t.to_dict("records")):
        if not chosen[i]:
            rows.append(r)
            continue
        piece = r["dur"] / k
        for j in range(k):
            p = dict(r)
            p["start"] = r["start"] + j * (piece + gap)
            p["end"] = p["start"] + piece
            p["dur"] = piece
            rows.append(p)
    return pd.DataFrame(rows)


def interrupt_ami(t, target, level, overlap_frac=0.3, max_gap=3.0, seed=0):
    """For a fraction `level` of eligible target turns, move the next other-gender turn
    earlier so that it starts inside the target turn. Durations and counts never change."""
    rng = np.random.default_rng(seed)
    t = t.sort_values(["start", "end"]).reset_index(drop=True)
    st, en = t["start"].to_numpy().copy(), t["end"].to_numpy().copy()
    dur, gen = t["dur"].to_numpy(), t["gender"].to_numpy()
    prev_end = t.groupby("speaker")["end"].shift().fillna(-np.inf).to_numpy()
    used, pairs = set(), []
    for i in np.flatnonzero((gen == target) & (dur >= 1.0)):
        ns = en[i] - overlap_frac * dur[i]
        for j in np.flatnonzero((st > en[i]) & (st <= en[i] + max_gap) & (gen != target)):
            if j in used:
                continue
            if ns >= prev_end[j] + 0.05:
                pairs.append((j, ns))
                used.add(j)
                break
    n_pick = int(round(level * len(pairs)))
    if n_pick > 0:
        for p in rng.choice(len(pairs), size=n_pick, replace=False):
            j, ns = pairs[p]
            st[j], en[j] = ns, ns + dur[j]
    out = t.copy()
    out["start"], out["end"] = st, en
    return out


def cluster_ami(t, target, level, seed=0):
    """Shrink the silences between each target speaker's own turns by (1 - level)."""
    t = t.sort_values(["start", "end"]).reset_index(drop=True)
    st, en = t["start"].to_numpy().copy(), t["end"].to_numpy().copy()
    for s in t.loc[t["gender"] == target, "speaker"].unique():
        idx = np.flatnonzero((t["speaker"] == s).to_numpy())
        d = en[idx] - st[idx]
        gaps = np.maximum(st[idx[1:]] - en[idx[:-1]], 0.0)
        new = np.empty(len(idx))
        new[0] = st[idx[0]]
        for k in range(1, len(idx)):
            new[k] = new[k - 1] + d[k - 1] + (1 - level) * gaps[k - 1]
        st[idx], en[idx] = new, new + d
    out = t.copy()
    out["start"], out["end"] = st, en
    return out


ops = {"O1_fragment": fragment_ami, "O2_interrupt": interrupt_ami, "O3_cluster": cluster_ami}
levels = np.round(np.linspace(0.1, 1.0, 10), 1)
meeting_turns = {m: turns[turns["meeting"] == m] for m in main["meeting"]}

# identity check: time parity = turns parity + length ratio
print("log_parity_time == log_parity_turns + log_len_ratio :",
      np.allclose(feat["log_parity_time"], feat["log_parity_turns"] + feat["log_len_ratio"]))

# invariance check on 20 meetings at the strongest level
chk = []
for m in list(meeting_turns)[:20]:
    t = meeting_turns[m]
    T0, N0 = t.groupby("gender")["dur"].sum(), t.groupby("gender").size()
    for name, fn in ops.items():
        a = fn(t, "F", 1.0)
        T1, N1 = a.groupby("gender")["dur"].sum(), a.groupby("gender").size()
        chk.append({"operator": name, "time_same": bool(np.allclose(T0, T1)),
                    "turns_same": bool((N0 == N1).all())})
print(pd.DataFrame(chk).groupby("operator").mean())

In [ ]:
rows = []
def add(m, op, sl, t):
    f = meeting_features(t)
    if f:
        rows.append({"meeting": m, "operator": op, "signed_level": sl, **f})

for m, t in meeting_turns.items():
    add(m, "baseline", 0.0, t)
    for name, fn in ops.items():
        for tgt, sign in [("F", 1), ("M", -1)]:
            for lv in levels:
                add(m, name, sign * lv, fn(t, tgt, lv))

aug = pd.DataFrame(rows)
aug["series"] = aug["meeting"].str[:-1]
aug["log_cut_rate"] = (np.log((aug["cut_F"] + 0.5) / aug["T_F"])
                       - np.log((aug["cut_M"] + 0.5) / aug["T_M"]))
aug.to_csv("../outputs/ami_augmented.csv", index=False)
print("Rows:", aug.shape, "| meetings:", aug["meeting"].nunique())

cols_m = ["log_parity_turns", "log_len_ratio", "interruption_asym", "log_cut_rate",
          "position_gap", "log_parity_time"]
base = aug[aug["operator"] == "baseline"].set_index("meeting")[cols_m]
sd = base.std()

print("\nSpearman between injected level and each feature:")
res = {n: {c: round(spearmanr(aug.loc[aug["operator"] == n, "signed_level"],
                              aug.loc[aug["operator"] == n, c])[0], 3) for c in cols_m}
       for n in ops}
print(pd.DataFrame(res).T.to_string())

print("\nMean change at level +1.0 / -1.0, in units of the baseline SD:")
eff = {}
for n in ops:
    for lv, tag in [(1.0, "F side"), (-1.0, "M side")]:
        s = aug[(aug["operator"] == n) & (aug["signed_level"] == lv)].set_index("meeting")[cols_m]
        eff[f"{n} ({tag})"] = ((s - base.loc[s.index]).mean() / sd).round(2)
print(pd.DataFrame(eff).T.to_string())

print("\nBaseline: Spearman of log_cut_rate with log_len_ratio:",
      round(spearmanr(base["log_cut_rate"], base["log_len_ratio"])[0], 2),
      "| (old interruption_asym:", round(spearmanr(base["interruption_asym"], base["log_len_ratio"])[0], 2), ")")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_predict
from scipy.stats import spearmanr

comp_map = feat.set_index("meeting")["component"]
d = aug.dropna(subset=cols_m).copy()
d["component"] = d["meeting"].map(comp_map)
d["label"] = 10 * d["signed_level"]
y = d["label"].to_numpy()
cv = GroupKFold(n_splits=5)

sets = {
    "B1 time parity only": ["log_parity_time"],
    "B2 turn parity only": ["log_parity_turns"],
    "A: turns,len,intasym,posgap": ["log_parity_turns", "log_len_ratio", "interruption_asym", "position_gap"],
    "B: turns,len,cutrate,posgap": ["log_parity_turns", "log_len_ratio", "log_cut_rate", "position_gap"],
    "C: A + cutrate": ["log_parity_turns", "log_len_ratio", "interruption_asym", "log_cut_rate", "position_gap"],
}

def fit_set(cols, groups):
    pipe = make_pipeline(StandardScaler(), PolynomialFeatures(include_bias=False), Ridge())
    grid = {"polynomialfeatures__degree": [1, 2], "ridge__alpha": [1, 10, 100, 1000]}
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="r2")
    gs.fit(d[cols].to_numpy(), y, groups=groups)
    return gs

models, rows = {}, []
for name, cols in sets.items():
    gs = fit_set(cols, d["series"])
    models[name] = gs
    rows.append({"feature set": name,
                 "CV R2 (by series)": round(gs.best_score_, 3),
                 "degree": gs.best_params_["polynomialfeatures__degree"],
                 "alpha": gs.best_params_["ridge__alpha"]})
print(pd.DataFrame(rows).to_string(index=False))

print()
print("Sensitivity: CV by component instead of series")
for name in ["A: turns,len,intasym,posgap", "B: turns,len,cutrate,posgap"]:
    g2 = fit_set(sets[name], d["component"])
    print("  ", name, "R2 =", round(g2.best_score_, 3))

cand = {k: models[k].best_score_ for k in list(sets)[2:]}
best = max(cand, key=cand.get)
print()
print("Best of A/B/C:", best)

cols = sets[best]
d["pred"] = cross_val_predict(models[best].best_estimator_, d[cols].to_numpy(), y,
                              cv=cv, groups=d["series"])

print()
print("Spearman(predicted score, injected level) per operator:")
for n, sub in d[d["operator"] != "baseline"].groupby("operator"):
    print("  ", n, round(spearmanr(sub["pred"], sub["signed_level"])[0], 3))

b = d.loc[d["operator"] == "baseline", "pred"]
print()
print("Baseline (unmodified) meetings: score mean =", round(b.mean(), 3), "| SD =", round(b.std(), 3))

tab = (d[d["operator"] != "baseline"]
       .pivot_table(index="signed_level", columns="operator", values="pred").round(2))
print(tab.loc[[-1.0, -0.5, -0.2, 0.2, 0.5, 1.0]].to_string())

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan

colsA = sets["A: turns,len,intasym,posgap"]
ds = d.sort_values(["meeting", "operator", "signed_level"]).reset_index(drop=True)
Xc = sm.add_constant(ds[colsA])

vif = pd.Series([variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])],
                index=colsA).round(2)
print("VIF (should be well below 10):")
print(vif.to_string())

ols = sm.OLS(ds["label"], Xc).fit()
print()
print("Linear fit, in-sample R2:", round(ols.rsquared, 3))
print("Durbin-Watson:", round(durbin_watson(ols.resid), 3))
print("Breusch-Pagan p-value:", round(het_breuschpagan(ols.resid, Xc)[1], 6))

def nested_cv(cols, groups):
    Xa = d[cols].to_numpy()
    g = groups.to_numpy()
    out = []
    for tr, te in GroupKFold(n_splits=5).split(Xa, y, g):
        pipe = make_pipeline(StandardScaler(), PolynomialFeatures(include_bias=False), Ridge())
        grid = {"polynomialfeatures__degree": [1, 2], "ridge__alpha": [1, 10, 100, 1000]}
        gs = GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=3), scoring="r2")
        gs.fit(Xa[tr], y[tr], groups=g[tr])
        out.append(gs.score(Xa[te], y[te]))
    return np.round(out, 3)

print()
print("Nested CV R2 by series (honest estimate):")
for name in ["B1 time parity only", "B2 turn parity only", "A: turns,len,intasym,posgap"]:
    s = nested_cv(sets[name], d["series"])
    print("  ", name, "| folds:", s.tolist(), "| mean:", round(float(s.mean()), 3))

In [ ]:
def spread_features(t):
    isF = (t["gender"] == "F").to_numpy()
    isM = (t["gender"] == "M").to_numpy()
    if isF.sum() < 3 or isM.sum() < 3:
        return None
    dur = t["dur"].to_numpy()
    st, en = t["start"].to_numpy(), t["end"].to_numpy()
    t0, t1 = st.min(), en.max()
    mid = ((st + en) / 2 - t0) / max(t1 - t0, 1e-9)

    def wsd(mask):
        w, m = dur[mask], mid[mask]
        c = (m * w).sum() / w.sum()
        return np.sqrt((w * (m - c) ** 2).sum() / w.sum())

    eps = 1e-9
    return {"len_sd_ratio": np.log((dur[isF].std(ddof=1) + eps) / (dur[isM].std(ddof=1) + eps)),
            "pos_sd_ratio": np.log((wsd(isF) + eps) / (wsd(isM) + eps))}

rows = []
def addv(m, op, sl, t):
    f = spread_features(t)
    if f:
        rows.append({"meeting": m, "operator": op, "signed_level": sl, **f})

for m, t in meeting_turns.items():
    addv(m, "baseline", 0.0, t)
    for name, fn in ops.items():
        for tgt, sign in [("F", 1), ("M", -1)]:
            for lv in levels:
                addv(m, name, sign * lv, fn(t, tgt, lv))

sp = pd.DataFrame(rows)
keys = ["meeting", "operator", "signed_level"]
so4 = d.merge(sp, on=keys, how="inner")
print("Rows:", len(so4), "of", len(d))

pairs = {"turn length": ("log_len_ratio", "len_sd_ratio"),
         "position": ("position_gap", "pos_sd_ratio")}
base_sd = so4[so4["operator"] == "baseline"][["log_len_ratio", "len_sd_ratio",
                                              "position_gap", "pos_sd_ratio"]].std()

print()
print("Spearman with injected level (by operator), and effect at +/-1.0 in baseline-SD units")
out = []
for op in ["O1_fragment", "O2_interrupt", "O3_cluster"]:
    sub = so4[so4["operator"] == op]
    one = so4[(so4["operator"] == op) & (so4["signed_level"] == 1.0)].set_index("meeting")
    b0 = so4[so4["operator"] == "baseline"].set_index("meeting")
    for what, (mean_c, sd_c) in pairs.items():
        for kind, c in [("mean", mean_c), ("SD", sd_c)]:
            eff = ((one[c] - b0.loc[one.index, c]).mean()) / base_sd[c]
            out.append({"operator": op, "feature": what, "form": kind,
                        "spearman": round(spearmanr(sub["signed_level"], sub[c])[0], 3),
                        "effect_in_SD": round(float(eff), 2)})
print(pd.DataFrame(out).to_string(index=False))

def cv_r2(cols):
    X, yy, g = so4[cols].to_numpy(), so4["label"].to_numpy(), so4["series"]
    pipe = make_pipeline(StandardScaler(), PolynomialFeatures(include_bias=False), Ridge())
    grid = {"polynomialfeatures__degree": [1, 2], "ridge__alpha": [1, 10, 100, 1000]}
    gs = GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=5), scoring="r2")
    gs.fit(X, yy, groups=g)
    return round(gs.best_score_, 3)

base_cols = ["log_parity_turns", "log_len_ratio", "interruption_asym", "position_gap"]
print()
print("Grouped CV R2 (by series):")
print("  A (means only)      :", cv_r2(base_cols))
print("  A + len SD          :", cv_r2(base_cols + ["len_sd_ratio"]))
print("  A + position SD     :", cv_r2(base_cols + ["pos_sd_ratio"]))
print("  A + both SD forms   :", cv_r2(base_cols + ["len_sd_ratio", "pos_sd_ratio"]))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

final_cols = ["log_parity_turns", "log_len_ratio", "interruption_asym", "position_gap", "pos_sd_ratio"]
yy = so4["label"].to_numpy()
grp = so4["series"].to_numpy()
XF = so4[final_cols].to_numpy()

pipe = make_pipeline(StandardScaler(), PolynomialFeatures(include_bias=False), Ridge())
grid = {"polynomialfeatures__degree": [1, 2], "ridge__alpha": [1, 10, 100, 1000]}
gsF = GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=5), scoring="r2")
gsF.fit(XF, yy, groups=grp)
deg = gsF.best_params_["polynomialfeatures__degree"]
alpha = gsF.best_params_["ridge__alpha"]
print("Chosen:", gsF.best_params_, "| grouped-CV R2:", round(gsF.best_score_, 3))

nested = []
for tr, te in GroupKFold(n_splits=5).split(XF, yy, grp):
    g = GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=3), scoring="r2")
    g.fit(XF[tr], yy[tr], groups=grp[tr])
    nested.append(g.score(XF[te], yy[te]))
print("Nested CV R2 by fold:", np.round(nested, 3).tolist(), "| mean:", round(float(np.mean(nested)), 3))

so4["pred"] = cross_val_predict(gsF.best_estimator_, XF, yy, cv=GroupKFold(n_splits=5), groups=grp)
cal = LinearRegression().fit(so4[["pred"]], yy)
a, b = float(cal.coef_[0]), float(cal.intercept_)
so4["score10"] = np.clip(a * so4["pred"] + b, -10, 10)
print("Calibration: score10 = %.2f * pred + %.2f" % (a, b))

bl = so4[so4["operator"] == "baseline"]["score10"]
print("Unmodified meetings: mean %.2f | SD %.2f" % (bl.mean(), bl.std()))
tab = (so4[so4["operator"] != "baseline"]
       .pivot_table(index="signed_level", columns="operator", values="score10").round(2))
print(tab.loc[[-1.0, -0.5, -0.2, 0.2, 0.5, 1.0]].to_string())

rng = np.random.default_rng(42)
series = np.array(sorted(so4["series"].unique()))
idx_by = {s: np.flatnonzero(grp == s) for s in series}
ops_list = ["O1_fragment", "O2_interrupt", "O3_cluster"]
op_arr = so4["operator"].to_numpy()
lvl_arr = so4["signed_level"].to_numpy()
boot_pipe = make_pipeline(StandardScaler(), PolynomialFeatures(degree=deg, include_bias=False), Ridge(alpha=alpha))

B = 500
r2s, coefs = [], []
sp = {k: [] for k in ops_list}
for _ in range(B):
    pick = rng.choice(series, size=len(series), replace=True)
    tr = np.concatenate([idx_by[s] for s in pick])
    oob = np.setdiff1d(np.arange(len(yy)), tr)
    if len(oob) == 0:
        continue
    boot_pipe.fit(XF[tr], yy[tr])
    p = boot_pipe.predict(XF[oob])
    r2s.append(r2_score(yy[oob], p))
    coefs.append(boot_pipe.named_steps["ridge"].coef_[:len(final_cols)])
    for k in ops_list:
        m = op_arr[oob] == k
        sp[k].append(spearmanr(p[m], lvl_arr[oob][m])[0])

def ci(v):
    return np.round(np.percentile(v, [2.5, 50, 97.5]), 3).tolist()

print()
print("Cluster bootstrap over series (%d resamples), out-of-bag series only:" % len(r2s))
print("  R2 [2.5%, median, 97.5%]:", ci(r2s))
for k in ops_list:
    print("  Spearman", k, ci(sp[k]))
coefs = np.array(coefs)
print("  Standardised linear coefficients [2.5%, median, 97.5%]:")
for j, c in enumerate(final_cols):
    print("   ", c, ci(coefs[:, j]))

draws = rng.integers(0, len(series), size=(2000, 10))
def dataset_scores(df):
    g = df.groupby("series")["score10"].agg(["sum", "count"]).reindex(series).fillna(0)
    s, c = g["sum"].to_numpy(), g["count"].to_numpy()
    return s[draws].sum(axis=1) / np.maximum(c[draws].sum(axis=1), 1)

bs = dataset_scores(so4[so4["operator"] == "baseline"])
thr = float(np.percentile(np.abs(bs), 95))
print()
print("Dataset-level score for an unmodified dataset of 10 series: mean %.2f | SD %.2f | 95%% band +/- %.2f" % (bs.mean(), bs.std(), thr))
rows = []
for k in ops_list:
    for lv in [0.2, 0.5, 1.0]:
        sc = dataset_scores(so4[(so4["operator"] == k) & (so4["signed_level"] == lv)])
        rows.append({"operator": k, "injected level": lv,
                     "mean dataset score": round(float(sc.mean()), 2),
                     "flagged outside band (%)": round(100 * float((np.abs(sc) > thr).mean()), 1)})
print(pd.DataFrame(rows).to_string(index=False))
so4.to_csv("../outputs/ami_so4_scored.csv", index=False)

In [ ]:
print("so4 columns:", list(so4.columns))
print("so4 shape:", so4.shape)
print(so4[so4["operator"] == "baseline"].head(3).T.to_string())

for name in ["feat", "meeting_turns", "turns", "intervals", "info"]:
    if name in globals():
        obj = globals()[name]
        print(name, type(obj).__name__, getattr(obj, "shape", None))
        if hasattr(obj, "columns"):
            print("   ", list(obj.columns))

dome = pd.read_csv("../data/AMI/dome_annotations_M1.csv", sep=";")
dome["name"] = dome["name"].str.strip()
print("DOME:", dome.shape, "meetings:", dome["name"].nunique())
print(dome.head(3).to_string())

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from scipy.stats import kruskal, friedmanchisquare, spearmanr

measures = {"B1 time parity": "log_parity_time",
            "B2 turn parity": "log_parity_turns",
            "EuqVid-Temporal score": "score10"}
base_rows = so4[so4["operator"] == "baseline"]
res = []
for op in ["O1_fragment", "O2_interrupt", "O3_cluster"]:
    sub = pd.concat([base_rows, so4[so4["operator"] == op]])
    for mname, col in measures.items():
        vals = sub[col].round(6)
        row = {"operator": op, "measure": mname}
        if vals.nunique() > 2:
            row["Spearman vs level"] = round(float(spearmanr(vals, sub["signed_level"])[0]), 3)
            groups = [g.to_numpy() for _, g in vals.groupby(sub["signed_level"])]
            H, p = kruskal(*groups)
            row["KW H"] = round(float(H), 1)
            row["KW p"] = float(p)
            row["epsilon2"] = round(float(H) / (len(vals) - 1), 3)
        else:
            row["Spearman vs level"], row["KW H"], row["KW p"], row["epsilon2"] = np.nan, np.nan, np.nan, np.nan
        try:
            wide = sub.assign(v=vals).pivot_table(index="meeting", columns="signed_level", values="v").dropna()
            chi, pf = friedmanchisquare(*[wide[c].to_numpy() for c in wide.columns])
            row["Friedman p"] = float(pf)
        except Exception:
            row["Friedman p"] = np.nan
        res.append(row)
tab2 = pd.DataFrame(res)
pd.set_option("display.float_format", lambda x: "%.3g" % x)
print(tab2.to_string(index=False))

In [ ]:
from sklearn.linear_model import LinearRegression

base = so4[so4["operator"] == "baseline"].reset_index(drop=True)
print("Real (unmodified) meetings:", len(base))
print("Spearman(score10, B1 time parity): %.3f" % spearmanr(base["score10"], base["log_parity_time"])[0])
lr = LinearRegression().fit(base[["log_parity_time"]], base["score10"])
print("Share of score10 variance explained by time parity alone: %.3f" % lr.score(base[["log_parity_time"]], base["score10"]))

TOL = 0.10
i, j = np.triu_indices(len(base), k=1)
tp = base["log_parity_time"].to_numpy()
sc = base["score10"].to_numpy()
ser = base["series"].to_numpy()
matched = (np.abs(tp[i] - tp[j]) <= TOL) & (ser[i] != ser[j])
far = (np.abs(tp[i] - tp[j]) > 1.0) & (ser[i] != ser[j])
d_m = np.abs(sc[i][matched] - sc[j][matched])
d_f = np.abs(sc[i][far] - sc[j][far])
print("Matched-total pairs (|diff time parity| <= %.2f, different series): %d" % (TOL, matched.sum()))
print("  |score diff| in matched pairs: median %.2f | 75%% %.2f | 90%% %.2f" % tuple(np.percentile(d_m, [50, 75, 90])))
if far.sum() > 0:
    print("  |score diff| in far-apart pairs (time parity differs by > 1): median %.2f" % np.median(d_f))
print("  Matched pairs with |score diff| > 2: %.1f%%" % (100 * (d_m > 2).mean()))

fcols = ["log_parity_turns", "log_len_ratio", "interruption_asym", "position_gap", "pos_sd_ratio"]
dscore = sc[i][matched] - sc[j][matched]
print("Which feature differences go with the score difference in matched pairs (Spearman):")
for c in fcols:
    v = base[c].to_numpy()
    print("   ", c, round(float(spearmanr(v[i][matched] - v[j][matched], dscore)[0]), 3))

order = np.argsort(-np.abs(sc[i][matched] - sc[j][matched]))[:5]
mi, mj = i[matched][order], j[matched][order]
show = ["meeting", "n_F", "n_M", "N_F", "N_M", "log_parity_time", "log_parity_turns", "log_len_ratio", "score10"]
print("Five most different matched pairs (same speaking-time parity, different score):")
for a, b in zip(mi, mj):
    print(base.loc[[a, b], show].round(2).to_string(index=False))
    print("---")

In [ ]:
import inspect
import xml.etree.ElementTree as ET
from scipy.stats import spearmanr

RANK1_IS_MOST_DOMINANT = True   # set this after reading the DOME documentation

# seat letter (A-D) -> person id, per meeting, from meetings.xml
root = ET.parse("../data/AMI/manual_1.6.2/corpusResources/meetings.xml").getroot()
seat_person = {}
for mt in root.iter("meeting"):
    obs = mt.get("observation")
    for sp in mt.findall("speaker"):
        seat_person[(obs, sp.get("nxt_agent"))] = sp.get("global_name")
print("seat->person entries:", len(seat_person), "| example:", list(seat_person.items())[:2])

dome = pd.read_csv("../data/AMI/dome_annotations_M1.csv", sep=";")
dome.columns = [c.strip() for c in dome.columns]
dome["name"] = dome["name"].str.strip()
dome = dome.rename(columns={"start (sec)": "t0", "end (sec)": "t1"})
seats = ["A", "B", "C", "D"]

mean_rank = np.mean([dome[["A%d%d" % (a, k) for k in range(1, 5)]].to_numpy() for a in (1, 2, 3)], axis=0)
person_gender = turns.groupby(["meeting", "speaker"])["gender"].first().to_dict()

rows = []
skipped = {"seat not mapped": 0, "gender unknown": 0, "single gender": 0, "empty slice": 0}
for r in range(len(dome)):
    m, t0, t1 = dome.loc[r, "name"], dome.loc[r, "t0"], dome.loc[r, "t1"]
    pers = {s: seat_person.get((m, s)) for s in seats}
    if any(v is None for v in pers.values()):
        skipped["seat not mapped"] += 1
        continue
    g = {s: person_gender.get((m, pers[s])) for s in seats}
    if sum(v in ("F", "M") for v in g.values()) < 4:
        skipped["gender unknown"] += 1
        continue
    nF = sum(v == "F" for v in g.values())
    nM = sum(v == "M" for v in g.values())
    if nF == 0 or nM == 0:
        skipped["single gender"] += 1
        continue
    rk = {s: mean_rank[r, k] for k, s in enumerate(seats)}
    rF = np.mean([rk[s] for s in seats if g[s] == "F"])
    rM = np.mean([rk[s] for s in seats if g[s] == "M"])
    gap = (rM - rF) if RANK1_IS_MOST_DOMINANT else (rF - rM)
    d = turns[(turns["meeting"] == m) & (turns["end"] > t0) & (turns["start"] < t1)].copy()
    d["cd"] = (d["end"].clip(upper=t1) - d["start"].clip(lower=t0)).clip(lower=0)
    TF = d.loc[d["gender"] == "F", "cd"].sum()
    TM = d.loc[d["gender"] == "M", "cd"].sum()
    st = d[(d["start"] >= t0) & (d["start"] < t1)]
    NF = (st["gender"] == "F").sum()
    NM = (st["gender"] == "M").sum()
    if min(TF, TM, NF, NM) <= 0:
        skipped["empty slice"] += 1
        continue
    rows.append({"meeting": m, "t0": t0, "t1": t1, "dome_gap": gap, "nF": nF, "nM": nM,
                 "b1_time": np.log((TF / nF) / (TM / nM)),
                 "b2_turns": np.log((NF / nF) / (NM / nM))})
print("Skipped slices:", skipped)
ds = pd.DataFrame(rows)
print("Usable mixed-gender DOME slices:", len(ds), "| meetings:", ds["meeting"].nunique() if len(ds) else 0)
if len(ds) > 5:
    rng = np.random.default_rng(1)
    ms = ds["meeting"].unique()
    for col in ["b1_time", "b2_turns"]:
        r0 = spearmanr(ds[col], ds["dome_gap"])[0]
        bs = []
        for _ in range(2000):
            pick = rng.choice(ms, size=len(ms), replace=True)
            dd = pd.concat([ds[ds["meeting"] == x] for x in pick])
            if dd[col].nunique() > 2 and dd["dome_gap"].nunique() > 2:
                bs.append(spearmanr(dd[col], dd["dome_gap"])[0])
        lo, hi = np.percentile(bs, [2.5, 97.5]) if len(bs) > 0 else (np.nan, np.nan)
        print("%s vs DOME gap: Spearman %.3f | meeting-bootstrap 95%% [%.2f, %.2f]" % (col, r0, lo, hi))
    ds.to_csv("../outputs/dome_slices_baselines.csv", index=False)

print(inspect.signature(meeting_features))
print(inspect.getsource(meeting_features)[:2500])

In [ ]:
import inspect
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr

cols4 = ["log_parity_turns", "log_len_ratio", "interruption_asym", "position_gap"]
dome_series = set(turns.loc[turns["meeting"].isin(ds["meeting"].unique()), "series"])
train = so4[~so4["series"].isin(dome_series)]
print("DOME series excluded from training:", sorted(dome_series), "| training rows:", len(train))
mdl = make_pipeline(StandardScaler(), PolynomialFeatures(degree=deg, include_bias=False), Ridge(alpha=alpha))
mdl.fit(train[cols4].to_numpy(), train["label"].to_numpy())

recs = []
for k, r in ds.iterrows():
    t = turns[(turns["meeting"] == r["meeting"]) & (turns["end"] > r["t0"]) & (turns["start"] < r["t1"])].copy()
    t["start"] = t["start"].clip(lower=r["t0"])
    t["end"] = t["end"].clip(upper=r["t1"])
    t["dur"] = t["end"] - t["start"]
    f = meeting_features(t)
    if f is not None and min(f["T_F"], f["T_M"]) > 0:
        recs.append(dict(f, idx=k))
sf = pd.DataFrame(recs).set_index("idx")
d4 = ds.loc[sf.index].copy()
d4["score4"] = mdl.predict(sf[cols4].to_numpy())
d4["b1"] = sf["log_parity_time"]
d4["b2"] = sf["log_parity_turns"]
for c in cols4:
    d4[c] = sf[c]
print("Slices scored:", len(d4), "| meetings:", d4["meeting"].nunique())
print("Slice score: mean %.2f | SD %.2f" % (d4["score4"].mean(), d4["score4"].std()))

def r_of(df, col):
    return spearmanr(df[col], df["dome_gap"])[0]

names = ["score4", "b1", "b2"]
point = {c: r_of(d4, c) for c in names}
rng = np.random.default_rng(7)
ms = d4["meeting"].unique()
rs = {c: [] for c in names}
diff = []
for _ in range(2000):
    pick = rng.choice(ms, size=len(ms), replace=True)
    bb = pd.concat([d4[d4["meeting"] == x] for x in pick])
    if bb["dome_gap"].nunique() < 3:
        continue
    vals = {c: r_of(bb, c) for c in names}
    if any(np.isnan(v) for v in vals.values()):
        continue
    for c in names:
        rs[c].append(vals[c])
    diff.append(vals["score4"] - vals["b1"])
print("Spearman with DOME gap (positive = agrees), meeting-bootstrap 95% CI:")
for c in names:
    lo, hi = np.percentile(rs[c], [2.5, 97.5])
    print("  %-7s %.3f [%.2f, %.2f]" % (c, point[c], lo, hi))
lo, hi = np.percentile(diff, [2.5, 97.5])
print("  score4 minus b1: %.3f [%.2f, %.2f]" % (point["score4"] - point["b1"], lo, hi))

print("Single features vs DOME gap (Spearman):")
for c in cols4:
    print("   ", c, round(float(r_of(d4, c)), 3))
d4.to_csv("../outputs/dome_slices_scored4.csv", index=False)

print(inspect.signature(spread_features))
print(inspect.getsource(spread_features)[:2500])

In [ ]:
from scipy.stats import rankdata

print("Identity check, max |parity_turns + len_ratio - parity_time|: %.2e" %
      float((so4["log_parity_turns"] + so4["log_len_ratio"] - so4["log_parity_time"]).abs().max()))
print("Largest change of log_parity_time inside one meeting across all variants: %.2e" %
      float(so4.groupby("meeting")["log_parity_time"].agg(lambda x: x.max() - x.min()).max()))

for df_ in (so4, d4):
    df_["struct_asym"] = (df_["log_parity_turns"] - df_["log_len_ratio"]) / 2
cols_s = ["struct_asym", "interruption_asym", "position_gap"]
train = so4[~so4["series"].isin(dome_series)]
ms_model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
ms_model.fit(train[cols_s].to_numpy(), train["label"].to_numpy())
d4["struct_score"] = ms_model.predict(d4[cols_s].to_numpy())

def incr(df):
    y = rankdata(df["dome_gap"])
    x1 = rankdata(df["b1"])
    x2 = rankdata(df["struct_score"])
    X = np.column_stack([np.ones(len(df)), (x1 - x1.mean()) / x1.std(), (x2 - x2.mean()) / x2.std()])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return beta[1], beta[2]

p_b1, p_st = incr(d4)
r_st = spearmanr(d4["struct_score"], d4["dome_gap"])[0]
rng = np.random.default_rng(11)
ms = d4["meeting"].unique()
bb1, bst, brs = [], [], []
for _ in range(2000):
    pick = rng.choice(ms, size=len(ms), replace=True)
    bb = pd.concat([d4[d4["meeting"] == x] for x in pick])
    if bb["dome_gap"].nunique() < 3 or bb["struct_score"].nunique() < 3:
        continue
    u, v = incr(bb)
    bb1.append(u)
    bst.append(v)
    brs.append(spearmanr(bb["struct_score"], bb["dome_gap"])[0])
print("Structure score alone vs DOME gap: Spearman %.3f [%.2f, %.2f]" % (r_st, *np.percentile(brs, [2.5, 97.5])))
print("Rank regression of DOME gap on [time parity b1, structure score] (standardised):")
print("  b1 coefficient       %.2f [%.2f, %.2f]" % (p_b1, *np.percentile(bb1, [2.5, 97.5])))
print("  structure coefficient %.2f [%.2f, %.2f]" % (p_st, *np.percentile(bst, [2.5, 97.5])))
d4.to_csv("../outputs/dome_slices_scored_struct.csv", index=False)

In [ ]:
import inspect
for fn in [build_turns, fragment_ami, interrupt_ami, cluster_ami]:
    print("=" * 70)
    print(fn.__name__, inspect.signature(fn))
    print(inspect.getsource(fn)[:3500])

In [ ]:
for name in ["GAP", "KEEP", "so4", "ops", "levels", "build_turns", "meeting_features", "spread_features"]:
    print(name, name in globals())

In [ ]:
import os
os.makedirs("../outputs/shared", exist_ok=True)

mi = pd.read_csv("../outputs/meld_intervals_words.csv")
mi["meeting"] = "S" + mi["season"].astype(str).str.zfill(2) + "E" + mi["episode"].astype(str).str.zfill(2)
mi["series"] = "MELD_S" + mi["season"].astype(str).str.zfill(2)
mi["corpus"] = "MELD"
mi["gender_source"] = "manual_map"
mi["component"] = "MELD_ALL"

cols = ["corpus", "meeting", "series", "speaker", "gender", "gender_source", "component"]
allp = pd.concat([turns[cols], mi[cols]], ignore_index=True)
allp["clip_id"] = allp["corpus"] + ":" + allp["meeting"]
allp["person_id"] = allp["corpus"] + ":" + allp["speaker"].astype(str)

gm = allp[["person_id", "gender", "gender_source", "component"]].drop_duplicates()
dup = gm["person_id"].duplicated(keep=False)
print("persons:", gm["person_id"].nunique(), "| persons with conflicting rows:", int(dup.sum()))
if dup.any():
    print(gm[dup].sort_values("person_id").head(10).to_string(index=False))
gm.to_csv("../outputs/shared/gender_map.csv", index=False)

rows = []
for cid, g in allp.groupby("clip_id"):
    pp = g.drop_duplicates("person_id")
    rows.append({"clip_id": cid, "corpus": g["corpus"].iloc[0], "meeting": g["meeting"].iloc[0],
                 "series": g["series"].iloc[0], "component": g["component"].iloc[0],
                 "n_F": int((pp["gender"] == "F").sum()), "n_M": int((pp["gender"] == "M").sum()),
                 "n_rows": len(g)})
clips = pd.DataFrame(rows)
clips["mixed_gender"] = (clips["n_F"] > 0) & (clips["n_M"] > 0)
print(clips.groupby("corpus")[["mixed_gender"]].agg(["count", "sum"]).to_string())
print(clips.head(3).to_string(index=False))
clips.to_csv("../outputs/shared/clips.csv", index=False)

In [ ]:
CFG = {"anchor_ratio": 4.0, "min_turns_per_clip": 20, "feature_version": "m2_v0.1"}
ANCHOR = float(np.log(CFG["anchor_ratio"]))

# ---- 1. MELD turns in exactly the same schema as the AMI turns ----
mi = pd.read_csv("../outputs/meld_intervals_words.csv")
mi["corpus"] = "MELD"
mi["meeting"] = "S" + mi["season"].astype(str).str.zfill(2) + "E" + mi["episode"].astype(str).str.zfill(2)
mi["series"] = "MELD_S" + mi["season"].astype(str).str.zfill(2)
mi["role"] = "NA"
mi["gender_source"] = "manual_map"
mi["component"] = "MELD_ALL"
mi["backchannel"] = mi["n_words"] <= 2
need = [c for c in turns.columns if c not in ("start", "end", "n_words", "dur")]
for c in need:
    if c not in mi.columns:
        mi[c] = "NA"
mturns = build_turns(mi[need + ["start", "end", "n_words", "backchannel"]].copy())
print("MELD utterances:", len(mi), "| kept after backchannel rule:", int((~mi["backchannel"]).sum()), "| turns:", len(mturns))

# ---- 2. features per MELD episode, with the same functions as AMI ----
rows = []
for m, t in mturns.groupby("meeting"):
    f = meeting_features(t)
    if f is None:
        continue
    s = spread_features(t)
    if s is None:
        s = {"len_sd_ratio": np.nan, "pos_sd_ratio": np.nan}
    rows.append({"corpus": "MELD", "meeting": m, "series": t["series"].iloc[0], "component": "MELD_ALL", **f, **s})
fm = pd.DataFrame(rows)
n_before = len(fm)
fm = fm[(fm["N_F"] + fm["N_M"]) >= CFG["min_turns_per_clip"]].copy()
print("MELD episodes with both genders:", n_before, "| with >= %d turns: %d" % (CFG["min_turns_per_clip"], len(fm)))

# ---- 3. AMI baseline rows (already computed) + a consistency check ----
fa = so4[so4["operator"] == "baseline"].copy()
fa["corpus"] = "AMI"
chk = []
for m in fa["meeting"].head(8):
    f = meeting_features(turns[turns["meeting"] == m])
    if f is not None:
        chk.append(abs(f["log_parity_turns"] - float(fa.loc[fa["meeting"] == m, "log_parity_turns"].iloc[0])))
print("Check: recomputed AMI log_parity_turns vs stored, max abs difference: %.2e" % max(chk))

common = ["corpus", "meeting", "series", "component", "n_F", "n_M", "N_F", "N_M", "T_F", "T_M",
          "log_parity_time", "log_parity_turns", "log_len_ratio", "interruption_asym",
          "position_gap", "len_sd_ratio", "pos_sd_ratio"]
base_all = pd.concat([fa[common], fm[common]], ignore_index=True)
base_all["clip_id"] = base_all["corpus"] + ":" + base_all["meeting"]
base_all["allocation_score"] = 10 * np.clip(base_all["log_parity_time"] / ANCHOR, -1, 1)
base_all["status_allocation"] = "valid"
base_all["status_structure"] = np.where(base_all["pos_sd_ratio"].isna(), "NA", "valid")
base_all["reason"] = np.where(base_all["pos_sd_ratio"].isna(), "fewer than 3 turns of one gender", "")
base_all["feature_version"] = CFG["feature_version"]

summ = base_all.groupby("corpus").agg(clips=("clip_id", "count"),
                                      alloc_mean=("allocation_score", "mean"),
                                      alloc_sd=("allocation_score", "std"),
                                      interruption_sd=("interruption_asym", "std"),
                                      structure_NA=("status_structure", lambda x: int((x == "NA").sum())))
print(summ.round(3).to_string())
share_zero = float((base_all.loc[base_all["corpus"] == "MELD", "interruption_asym"] == 0).mean())
print("MELD clips with interruption_asym exactly 0: %.1f%%" % (100 * share_zero))
base_all.to_csv("../outputs/features_all_baseline.csv", index=False)
print("saved ../outputs/features_all_baseline.csv", base_all.shape)

In [ ]:
print("GAP:", GAP, "| KEEP:", KEEP)
print("ops:", ops if not callable(ops) else "callable", "| levels:", levels)
for i, src in enumerate(In):
    if "levels" in src and "ops" in src and i < len(In) - 1:
        print("#### cell", i)
        print(src[:3500])

In [ ]:
from scipy.stats import spearmanr

# ---- 1. MELD augmentation: the SAME operators, levels and target rule as AMI ----
mt = {m: mturns[mturns["meeting"] == m] for m in fm["meeting"]}
rows = []
def add_meld(m, op, sl, t):
    f = meeting_features(t)
    if f is None:
        return
    s = spread_features(t)
    if s is None:
        s = {"len_sd_ratio": np.nan, "pos_sd_ratio": np.nan}
    rows.append({"corpus": "MELD", "meeting": m, "operator": op, "signed_level": sl, **f, **s})

for m, t in mt.items():
    add_meld(m, "baseline", 0.0, t)
    for name, fn in ops.items():
        for tgt, sign in [("F", 1), ("M", -1)]:
            for lv in levels:
                add_meld(m, name, sign * lv, fn(t, tgt, lv))
am = pd.DataFrame(rows)
am["series"] = am["meeting"].map(fm.set_index("meeting")["series"])
am["component"] = "MELD_ALL"
am["label"] = 10 * am["signed_level"]
print("MELD augmented rows:", am.shape, "| episodes:", am["meeting"].nunique())

# ---- 2. pool with the AMI table ----
sa = so4.copy()
sa["corpus"] = "AMI"
print("AMI label == 10 * signed_level:", bool(np.allclose(sa["label"], 10 * sa["signed_level"])))
keep = ["corpus", "meeting", "operator", "signed_level", "series", "component", "n_F", "n_M", "N_F", "N_M",
        "T_F", "T_M", "log_parity_time", "log_parity_turns", "log_len_ratio", "interruption_asym",
        "position_gap", "len_sd_ratio", "pos_sd_ratio", "label"]
aug_all = pd.concat([sa[keep], am[keep]], ignore_index=True)
aug_all["clip_id"] = aug_all["corpus"] + ":" + aug_all["meeting"]
aug_all["variant_id"] = (aug_all["clip_id"] + "|" + aug_all["operator"] + "|"
                         + aug_all["signed_level"].map(lambda x: "%+.2f" % x))
print("variant_id unique:", bool(aug_all["variant_id"].is_unique), "| rows:", len(aug_all),
      "| clips:", aug_all["clip_id"].nunique())

# ---- 3. checks ----
b = aug_all[aug_all["operator"] == "baseline"].set_index("clip_id")["log_parity_time"]
dev = (aug_all["log_parity_time"].to_numpy() - b.reindex(aug_all["clip_id"]).to_numpy())
print("max change of log_parity_time vs the clip's baseline (should be ~0): %.2e" % np.nanmax(np.abs(dev)))
print("rows with missing pos_sd_ratio:", int(aug_all["pos_sd_ratio"].isna().sum()))

tgt_feat = {"O1_fragment": "log_parity_turns", "O2_interrupt": "interruption_asym", "O3_cluster": "position_gap"}
res = {}
for corp in ["AMI", "MELD"]:
    res[corp] = {}
    for op, c in tgt_feat.items():
        s = aug_all[(aug_all["corpus"] == corp) & (aug_all["operator"] == op)]
        res[corp][op + " -> " + c] = round(float(spearmanr(s["signed_level"], s[c])[0]), 3)
print("Spearman between injected level and its target feature:")
print(pd.DataFrame(res).to_string())
aug_all.to_csv("../outputs/aug_all.csv", index=False)
print("saved ../outputs/aug_all.csv", aug_all.shape)

# ---- 4. flag the clips used in the main analysis ----
clips = pd.read_csv("../outputs/shared/clips.csv")
clips["in_main_analysis"] = clips["clip_id"].isin(set(base_all["clip_id"]))
print(clips.groupby("corpus")["in_main_analysis"].sum().to_string())
clips.to_csv("../outputs/shared/clips.csv", index=False)

In [ ]:
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

d5 = aug_all.copy()
d5["struct_asym"] = (d5["log_parity_turns"] - d5["log_len_ratio"]) / 2
cols_s = ["struct_asym", "interruption_asym", "position_gap", "pos_sd_ratio"]
d5 = d5.dropna(subset=cols_s).reset_index(drop=True)
d5["fold_group"] = np.where(d5["corpus"] == "AMI", d5["component"], d5["series"])

# ---- shared folds file: one fold per clip, identity-safe groups ----
cl = d5.drop_duplicates("clip_id")[["clip_id", "corpus", "series", "component", "fold_group"]].reset_index(drop=True)
cl["fold"] = -1
for k, (_, te) in enumerate(GroupKFold(n_splits=5).split(cl, groups=cl["fold_group"])):
    cl.loc[te, "fold"] = k
cl.to_csv("../outputs/shared/folds.csv", index=False)
d5 = d5.merge(cl[["clip_id", "fold"]], on="clip_id", how="left")
print("clips per fold:", cl["fold"].value_counts().sort_index().to_dict())

X = d5[cols_s].to_numpy()
y = d5["label"].to_numpy()
g = d5["fold_group"].to_numpy()
pipe = make_pipeline(StandardScaler(), Ridge())
grid = {"ridge__alpha": [1, 10, 100, 1000]}

# ---- pooled nested CV with the shared folds ----
pred = np.full(len(d5), np.nan)
for k in range(5):
    tr = (d5["fold"] != k).to_numpy()
    te = ~tr
    gs = GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=3), scoring="r2")
    gs.fit(X[tr], y[tr], groups=g[tr])
    pred[te] = gs.predict(X[te])
d5["pred"] = pred
print("Pooled nested CV R2: %.3f" % r2_score(y, pred))
for c in ["AMI", "MELD"]:
    m = (d5["corpus"] == c).to_numpy()
    print("  within %s: R2 %.3f" % (c, r2_score(y[m], pred[m])))

# ---- leave-one-corpus-out ----
ops_list = ["O1_fragment", "O2_interrupt", "O3_cluster"]
for src, dst in [("AMI", "MELD"), ("MELD", "AMI")]:
    tr = (d5["corpus"] == src).to_numpy()
    te = ~tr
    gs = GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=3), scoring="r2")
    gs.fit(X[tr], y[tr], groups=g[tr])
    p = gs.predict(X[te])
    sp = []
    for op in ops_list:
        mm = (d5.loc[te, "operator"] == op).to_numpy()
        sp.append("%s %.2f" % (op, spearmanr(y[te][mm], p[mm])[0]))
    print("train %s -> test %s: R2 %.3f | Spearman by operator: %s" % (src, dst, r2_score(y[te], p), "; ".join(sp)))

# ---- null-relative z-scores (unmodified clips define 'no structural bias') ----
base_mask = (d5["operator"] == "baseline").to_numpy()
mu_pool, sd_pool = d5.loc[base_mask, "pred"].mean(), d5.loc[base_mask, "pred"].std()
print("Unmodified clips, structure prediction:")
print(d5[base_mask].groupby("corpus")["pred"].agg(["count", "mean", "std"]).round(2).to_string())
print("pooled mean %.2f | pooled SD %.2f" % (mu_pool, sd_pool))
d5["z"] = (d5["pred"] - mu_pool) / sd_pool

rows = []
for corp in ["AMI", "MELD"]:
    nb = d5[(d5["corpus"] == corp) & base_mask]
    rows.append({"corpus": corp, "operator": "none (unmodified)", "|level|": 0.0,
                 "clips flagged |z|>1.96 (%)": round(100 * float((nb["z"].abs() > 1.96).mean()), 1)})
    for op in ops_list:
        for lv in [0.2, 0.5, 1.0]:
            s = d5[(d5["corpus"] == corp) & (d5["operator"] == op) & (d5["signed_level"].abs() == lv)]
            rows.append({"corpus": corp, "operator": op, "|level|": lv,
                         "clips flagged |z|>1.96 (%)": round(100 * float((s["z"].abs() > 1.96).mean()), 1)})
print("Clip-level detection:")
print(pd.DataFrame(rows).to_string(index=False))

# ---- dataset-level detection with 30-clip shards drawn from the pooled clips ----
clip_ids = np.array(sorted(d5["clip_id"].unique()))
pos = {c: i for i, c in enumerate(clip_ids)}
def clip_vec(df):
    v = np.full(len(clip_ids), np.nan)
    v[df["clip_id"].map(pos).to_numpy()] = df["z"].to_numpy()
    return v
rng = np.random.default_rng(42)
draws = np.array([rng.choice(len(clip_ids), size=30, replace=False) for _ in range(2000)])
null = np.nanmean(clip_vec(d5[base_mask])[draws], axis=1)
thr = float(np.percentile(np.abs(null), 95))
print("Dataset of 30 unmodified clips: mean z %.2f | SD %.2f | 95%% band +/- %.2f" % (null.mean(), null.std(), thr))
rows = []
for op in ops_list:
    for lv in [-1.0, -0.5, -0.2, 0.2, 0.5, 1.0]:
        s = d5[(d5["operator"] == op) & (d5["signed_level"].round(2) == lv)]
        sc = np.nanmean(clip_vec(s)[draws], axis=1)
        rows.append({"operator": op, "signed level": lv, "mean shard z": round(float(sc.mean()), 2),
                     "flagged outside band (%)": round(100 * float((np.abs(sc) > thr).mean()), 1)})
print(pd.DataFrame(rows).to_string(index=False))
d5.to_csv("../outputs/aug_all_scored.csv", index=False)

In [ ]:
import json

# ---- 1. final model on all pooled data ----
gsF = GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=5), scoring="r2")
gsF.fit(X, y, groups=g)
alpha_f = gsF.best_params_["ridge__alpha"]
sc = gsF.best_estimator_.named_steps["standardscaler"]
rg = gsF.best_estimator_.named_steps["ridge"]
print("Final alpha:", alpha_f, "| grouped-CV R2 used for selection: %.3f" % gsF.best_score_)
print("Standardised coefficients:", {c: round(float(v), 2) for c, v in zip(cols_s, rg.coef_)})

# ---- 2. cluster bootstrap over fold groups (V5) ----
rng = np.random.default_rng(7)
groups_u = np.unique(g)
idx_by = {k: np.flatnonzero(g == k) for k in groups_u}
r2b, cb = [], []
for _ in range(500):
    pick = rng.choice(groups_u, size=len(groups_u), replace=True)
    tr = np.concatenate([idx_by[k] for k in pick])
    oob = np.setdiff1d(np.arange(len(y)), tr)
    if len(oob) == 0:
        continue
    m_ = make_pipeline(StandardScaler(), Ridge(alpha=alpha_f)).fit(X[tr], y[tr])
    r2b.append(r2_score(y[oob], m_.predict(X[oob])))
    cb.append(m_.named_steps["ridge"].coef_)
cb = np.array(cb)
def ci(v):
    return np.round(np.percentile(v, [2.5, 50, 97.5]), 3).tolist()
print("Bootstrap over %d groups, out-of-bag R2 [2.5%%, median, 97.5%%]: %s" % (len(groups_u), ci(r2b)))
for j, c in enumerate(cols_s):
    print("  coefficient", c, ci(cb[:, j]))

# ---- 3. freeze: save every number needed to score a new clip ----
model = {"feature_version": CFG["feature_version"], "features": cols_s,
         "scaler_mean": sc.mean_.tolist(), "scaler_scale": sc.scale_.tolist(),
         "coef": rg.coef_.tolist(), "intercept": float(rg.intercept_), "alpha": float(alpha_f),
         "null_mean": float(mu_pool), "null_sd": float(sd_pool),
         "allocation_anchor_ratio": CFG["anchor_ratio"], "tiers_abs_z": [1.96, 3, 5],
         "trained_on": {"AMI_clips": int(d5.loc[d5["corpus"] == "AMI", "clip_id"].nunique()),
                        "MELD_clips": int(d5.loc[d5["corpus"] == "MELD", "clip_id"].nunique()),
                        "rows": int(len(d5))}}
with open("../outputs/m2_model_v0.1.json", "w") as fh:
    json.dump(model, fh, indent=2)

def score_from_json(df, m):
    Xz = (df[m["features"]].to_numpy() - np.array(m["scaler_mean"])) / np.array(m["scaler_scale"])
    return Xz @ np.array(m["coef"]) + m["intercept"]
diff = score_from_json(d5, model) - gsF.predict(X)
print("JSON scoring reproduces the fitted model: max abs difference %.2e" % np.abs(diff).max())
print("saved ../outputs/m2_model_v0.1.json")

In [ ]:
print("EPS:", EPS)

In [ ]:
import os, sys, importlib, glob

print("cwd:", os.getcwd())
hits = [p for p in glob.glob(os.path.join(os.getcwd(), "**", "m2_temporal.py"), recursive=True)]
hits += glob.glob(os.path.join(os.path.dirname(os.getcwd()), "**", "m2_temporal.py"), recursive=True)
hits = sorted(set(hits))
print("found:", hits)

In [ ]:
import sys, importlib, os
src_dir = os.path.abspath("../src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
importlib.invalidate_caches()
import m2_temporal as m2
importlib.reload(m2)
print("module loaded from:", m2.__file__)

ami_clips = d5[(d5["corpus"] == "AMI") & (d5["operator"] == "baseline")]["meeting"].head(5).tolist()
worst = 0.0
for mt in ami_clips:
    t = turns[turns["meeting"] == mt]
    f = m2.clip_features(t)
    ref = d5[(d5["clip_id"] == "AMI:" + mt) & (d5["operator"] == "baseline")].iloc[0]
    worst = max(worst, max(abs(f[c] - ref[c]) for c in m2.STRUCT_FEATURES))
    for op in m2.OPERATORS:
        for sl in (0.5, -0.5):
            f = m2.clip_features(m2.make_variant(t, op, sl))
            ref = d5[(d5["clip_id"] == "AMI:" + mt) & (d5["operator"] == op)
                     & (d5["signed_level"].round(2) == sl)].iloc[0]
            worst = max(worst, max(abs(f[c] - ref[c]) for c in m2.STRUCT_FEATURES))
print("max difference between the module and the notebook results:", worst)

In [ ]:
from scipy.stats import spearmanr, rankdata
assert "ds" in globals() and "dome_series" in globals(), "run the DOME cells (Cell I and Cell J) first"

# frozen specification (4 structure features, alpha from the final fit), refit WITHOUT the DOME series
train7 = d5[~d5["series"].isin(dome_series)]
mdl7 = make_pipeline(StandardScaler(), Ridge(alpha=alpha_f))
mdl7.fit(train7[cols_s].to_numpy(), train7["label"].to_numpy())
print("Training clips without DOME series:", train7["clip_id"].nunique(), "| DOME series excluded:", sorted(dome_series))

recs = []
for k, r in ds.iterrows():
    t = turns[(turns["meeting"] == r["meeting"]) & (turns["end"] > r["t0"]) & (turns["start"] < r["t1"])].copy()
    t["start"] = t["start"].clip(lower=r["t0"])
    t["end"] = t["end"].clip(upper=r["t1"])
    t["dur"] = t["end"] - t["start"]
    f = meeting_features(t)
    if f is None or min(f["T_F"], f["T_M"]) <= 0:
        continue
    s = spread_features(t)
    rec = {"idx": k, **f}
    rec["pos_sd_ratio"] = s["pos_sd_ratio"] if s else np.nan
    recs.append(rec)
sf7 = pd.DataFrame(recs).set_index("idx")
d7 = ds.loc[sf7.index, ["meeting", "t0", "t1", "dome_gap"]].copy()
for c in ["log_parity_time", "log_parity_turns", "log_len_ratio", "interruption_asym", "position_gap", "pos_sd_ratio"]:
    d7[c] = sf7[c]
d7["struct_asym"] = (d7["log_parity_turns"] - d7["log_len_ratio"]) / 2
d7["allocation"] = 10 * np.clip(d7["log_parity_time"] / np.log(CFG["anchor_ratio"]), -1, 1)
ok = d7[cols_s].notna().all(axis=1)
d7["struct_raw"] = np.nan
d7.loc[ok, "struct_raw"] = mdl7.predict(d7.loc[ok, cols_s].to_numpy())
print("DOME slices: %d with allocation | %d with structure score | meetings %d" %
      (len(d7), int(ok.sum()), d7["meeting"].nunique()))

def r_of(df, col):
    return spearmanr(df[col], df["dome_gap"])[0]

def incr(df):
    y = rankdata(df["dome_gap"])
    x1 = rankdata(df["allocation"])
    x2 = rankdata(df["struct_raw"])
    X2 = np.column_stack([np.ones(len(df)), (x1 - x1.mean()) / x1.std(), (x2 - x2.mean()) / x2.std()])
    return np.linalg.lstsq(X2, y, rcond=None)[0][1:]

d7s = d7[ok]
rng = np.random.default_rng(21)
res = {"allocation (all slices)": [], "allocation (structure subset)": [], "structure": [], "b_alloc": [], "b_struct": []}
for _ in range(2000):
    pick = rng.choice(d7["meeting"].unique(), size=d7["meeting"].nunique(), replace=True)
    bb = pd.concat([d7[d7["meeting"] == x] for x in pick])
    bs = bb.dropna(subset=["struct_raw"])
    if bb["dome_gap"].nunique() < 3 or bs["dome_gap"].nunique() < 3 or bs["struct_raw"].nunique() < 3:
        continue
    res["allocation (all slices)"].append(r_of(bb, "allocation"))
    res["allocation (structure subset)"].append(r_of(bs, "allocation"))
    res["structure"].append(r_of(bs, "struct_raw"))
    u, v = incr(bs)
    res["b_alloc"].append(u)
    res["b_struct"].append(v)
def ci(v):
    return np.percentile(v, [2.5, 97.5])
print("FINAL DOME CHECK (run once). Spearman with DOME gap, meeting-bootstrap 95% CI:")
pt = {"allocation (all slices)": r_of(d7, "allocation"), "allocation (structure subset)": r_of(d7s, "allocation"),
      "structure": r_of(d7s, "struct_raw")}
for k_, v_ in pt.items():
    lo, hi = ci(res[k_])
    print("  %-30s %.3f [%.2f, %.2f]" % (k_, v_, lo, hi))
pa, ps = incr(d7s)
print("Rank regression on the structure subset (standardised):")
print("  allocation coefficient %.2f [%.2f, %.2f]" % (pa, *ci(res["b_alloc"])))
print("  structure coefficient  %.2f [%.2f, %.2f]" % (ps, *ci(res["b_struct"])))
d7.to_csv("../outputs/dome_final_check.csv", index=False)